<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/rr_respiratory_mortality/notebooks/05.RR_Respiratory_Mortality_Spatial_Association.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Spatial Regression Analysis of Predicted Asthma Mortality and Relative Risk of Respiratory Mortality in Argentina (2022): SAR and SEM Implementation

In this notebook we implement spatial models to evaluate the spatial association between the predicted asthma mortality rate  and the RR of respiratory mortality across Argentine departments in 2022.

Because departments are spatially structured areal units, classical OLS regression may produce biased or inefficient estimates if spatial dependence exists. Our goal is to determine if spatial autocorrelation can manifest either:

* In the dependent variable (spatial spillover effects) modeled using a Spatial Autoregressive Model (SAR).

* In the error term (unobserved spatial processes) modeled using a Spatial Error Model (SEM).

The workflow follows a standard spatial medelling strategy:

1. Construct spatial weights (KNN-5)

2. Estimate OLS (Ordinary Least Squares) baseline model

3. Test residual spatial autocorrelation (Moran’s I)

4. Apply Lagrange Multiplier (LM) diagnostics

5. Estimate SAR and SEM models

6. Compare SAR and SEM Models

7. Mapping Spatial Model Prediction (SAR vs SEM)

8. Mapping Spatial Model Residuals (SAR vs SEM)

## 🤖 Load libraries

The libraries required for the analysis will be loaded

In [ ]:
# dataframe libraries
import pandas as pd

# Import NumPy for numerical and mathematical operations
import numpy as np

# geospatial libraries
import geopandas as gpd

# spatial modeling libraries
from spreg import OLS  # PySAL OLS regression
from spreg import ML_Lag, ML_Error

# geostatistical libraries
import pysal
import mapclassify
from libpysal.weights import Queen, KNN
from pysal.lib import weights
import esda
from esda.moran import Moran
from esda.moran import Moran_Local
import libpysal

# plot libraries
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import seaborn as sns


# other libraries
import os
from itables import init_notebook_mode, show

In [ ]:
# change directory to work folder (at the begining docker container enter into /home/jovyan/)
%cd work

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

##🗂 Load and prepare the dataset

In [ ]:
# open gdf data
gdf = gpd.read_file("pdt/rr_respiratory_mortality/data/gpkg/gdf_rrrm.gpkg")

In [ ]:
# visualize geodataframe
init_notebook_mode(all_interactive=True)
gdf

In [ ]:
# visualize column names
gdf.columns

In [ ]:
# select 'IDDPTO', 'AMR_PRED_2022' 'RRRM_2022_L' 'RRRM_2022_C', 'RRRM_2022_U', 'geometry' columns
gdf_ca_rrrm = gdf[[ 'IDDPTO', 'AMR_PRED_2022', 'RRRM_2022_L', 'RRRM_2022_C', 'RRRM_2022_U', 'geometry']]

In [ ]:
# visualize gdf
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm

In [ ]:
# original length of data
len(gdf_ca_rrrm)

In [ ]:
# drop all rows containing NaN for all rows
gdf_ca_rrrm = gdf_ca_rrrm.dropna().reset_index(drop=True)

In [ ]:
# length of data after dropping NaN
len(gdf_ca_rrrm)

In [ ]:
# visualize gdf
init_notebook_mode(all_interactive=True)
gdf_ca_rrrm

## 🏗️ Construct the Queen contiguity matrix



In [ ]:
# project gdf_ca_rrrm to 5347
gdf_ca_rrrm_pj= gdf_ca_rrrm.to_crs(5347)

In [ ]:
# Calculate Queen contiguity matrix
w_queen = weights.contiguity.Queen.from_dataframe(gdf_ca_rrrm_pj, use_index=True)

In [ ]:
# Check for isolated departments (no neighbors)
print("Number of islands (Queen):", len(w_queen.islands))

In [ ]:
# transform raw weights into row-standardized
w_queen.transform = 'R'

### Mapping Queen contiguity matrix

In [ ]:
# select all NaN rows for gdf
gdf_nan = gdf[gdf.isna().any(axis=1)]

In [ ]:
# from gdf_nan select IDDPTO and geometry
gdf_nan = gdf_nan[['IDDPTO', 'geometry']]

In [ ]:
# reproject gdf_nan
gdf_nan_pj = gdf_nan.to_crs(5347)

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(6, 8))

# Queen contiguity matrix
gdf_ca_rrrm_pj.plot(ax=ax)
w_queen.plot(
    gdf_ca_rrrm_pj,
    edge_kws=dict(linewidth=1, color="orangered"),
    node_kws=dict(marker="*"),
    ax=ax,
)
# NaN departments, where asthma mortality did not ocurr
gdf_nan_pj.plot(ax=ax, color="none", edgecolor="black", linewidth=0.15)
ax.set_axis_off()
ax.set_title("Queen contiguity matrix")

f.tight_layout()
plt.show()

## 🧱 Estimate OLS Baseline Model

Before estimating spatial models (SAR or SEM), we must establish a non-spatial baseline model using OLS.

The baseline regression is:

\begin{align*}
pamr_i= \beta_0+\beta_1 rrrm\_c_i + \epsilon_i
\end{align*}

Where:

* $pamr_i$: Predicted Asthma Mortality Rate in department $i$
* $rrrm\_c_i$: RR of Respiratory Mortality (central bound). For this analysis, we consider only the central bound as spatial autocorrelation is consistent across all bounds.   
* $\epsilon_i$: Random error term

OLS allows us to:

1. Establish the basic association between variables.

2. Evaluate coefficient magnitude and statistical significance.

3. Obtain residuals.

4. Test residual spatial autocorrelation (Moran’s i).

5. Provide a benchmark for AIC/BIC comparison with SAR and SEM.

If residuals are spatially autocorrelated, OLS assumptions are violated (Gauss–Markov conditions), spatial models become necessary.


In [ ]:
# Define dependent variable (y)
# reshape(-1,1) ensures a 2D column vector required by spreg
y = gdf_ca_rrrm_pj["AMR_PRED_2022"].values.reshape(-1, 1)

In [ ]:
# Define independent variable (X)
# Extract RR of respiratory mortality (central estimate)
X_var = gdf_ca_rrrm_pj["RRRM_2022_C"].values.reshape(-1, 1)

# Create intercept term (column of ones)
# Number of rows must match number of departments
intercept = np.ones((X_var.shape[0], 1))

# Combine intercept and independent variable horizontally
# First column = constant, second column = RRRM_2022_C
X = np.hstack((intercept, X_var))

In [ ]:
# Fit OLS regression
ols_model = OLS(
    y,                   # dependent variable
    X,                   # independent variables (including intercept)
    name_y="AMR_PRED_2022",
    name_x=["Constant", "RRRM_2022_C"]
)

In [ ]:
# Print regression summary
print(ols_model.summary)

In [ ]:
# Set up table of regression coefficients and p-value
init_notebook_mode(all_interactive=True)
pd.DataFrame(
    {
        # Pull out regression coefficients and
        # flatten as they are returned as Nx1 array
        "Coeff.": ols_model.betas.flatten(),
        # Pull out and flatten standard errors
        "Std. Error": ols_model.std_err.flatten(),
        # Pull out P-values from t-stat object
        "P-Value": [i[1] for i in ols_model.t_stat],
    },
    index=ols_model.name_x,
)

In [ ]:
# Extract important diagnostics for comparison
# R-squared
print("R-squared:", round(ols_model.r2, 4))

# AIC
print("AIC:", round(ols_model.aic, 2))

# BIC
print("BIC:", round(ols_model.schwarz, 2))

# Store residuals for Moran's I test in Step 5
ols_residuals = ols_model.u

The results show:

* The coefficient for the respiratory mortality RR (central bound) is not statistically significant (p ≈ 0.519), indicating no detectable linear association between the relative risk of respiratory mortality and the predicted asthma mortality rate. This result confirms the findings from the statistical correlation analysis.

* The intercept is also non-significant (p ≈ 0.1608)

* R² = 0.0025, meaning the explanatory variable accounts for less than 0.25% of the variance in the predicted asthma mortality. This is essentially zero explanatory power.

## ✔️ Conclusion

The standard spatial modelling workflow — OLS → Moran's I → LM diagnostics → SAR/SEM presupposes a theoretically and statistically meaningful baseline regression. If the OLS model lacks explanatory power, any spatial autocorrelation detected in the residuals (via Moran’s I) would be largely attributable to the dependent variable (predicted asthma mortality rate). Therefore, we would not be able to identify a meaningful relationship between the predicted asthma mortality rate and the RR of respiratory mortality.